In [1]:
import pandas as pd
import os
import shutil
from pathlib import Path
from tqdm import tqdm

In [6]:
pd.set_option('display.max_columns', None)

In [25]:
file_path = "P:/Dataset/MO-DBT-eligible/mo_cancer_cohort_focus.xlsx" 

df = pd.read_excel(file_path, engine='openpyxl')

# 2. Re-create target path logic from the notebook
root_path = 'P:/Dataset/MO-DBT-eligible'
SEP = os.sep

# We need to filter and construct paths exactly like the notebook did
df['PATIENT_STUDY_ID'] = df['PATIENT_STUDY_ID'].astype(str, errors='ignore')
series = df['side'].fillna('') + df['view'].fillna('') + '_' + df['series'].fillna('')

df['p_file_path'] = root_path + SEP + df['subtype'] + SEP + df['PATIENT_STUDY_ID'] + SEP + series

# Filter for rows that actually have a target path (matching notebook logic)
df_moved = df[~df['subtype'].isna() & ~df['side'].isna() & ~df['view'].isna()]

In [26]:
df_moved = df_moved.rename(columns={'folder_path': 'j_file_path'})

In [27]:
df_moved.reset_index(inplace=True, drop=True)

In [30]:
df_moved['j_file_path'].iloc[0], df_moved['p_file_path'].iloc[0]

('J:/Testing/jlee/MO_DBT/DICOM_data_and_summaries_882GB/Patient_4330344296/20191220/Study_SCREENING_MAMMOGRAPHY_DIGITAL_BILATERAL_W_TOMOSYN_62861703/Series_71300000_R_MLO_Intelligent_2D',
 'P:/Dataset/MO-DBT-eligible\\Luminal\\4330344296\\RMLO_IN2D')

In [31]:

print(f"Planning to recover files for {len(df_moved)} folders...")

for _, row in tqdm(df_moved.iterrows(), total=len(df_mo_cancer), desc='Recovering files'):
    source_dir = Path(row['p_file_path'])  # Where they are NOW (P:)
    target_dir = Path(row['j_file_path'])  # Where they belong (J:)
            
    if not source_dir.is_dir():
        # If the source doesn't exist, maybe it was already moved or never created
        continue

    if not target_dir.is_dir():
        target_dir.mkdir(parents=True, exist_ok=True)
        
    # Move everything from the current (P:) folder back to the original (J:) folder
    for item in source_dir.iterdir():
        current_item = item
        original_location = target_dir / item.name
        
        try:
            shutil.copy2(str(current_item), str(original_location))
        except Exception as e:
            print(f"  ERROR moving {item.name} back to {target_dir}: {e}")
    
    # # Optionally remove the now-empty directory on P:
    # try:
    #     if not any(source_dir.iterdir()):
    #         source_dir.rmdir()
    # except:
    #     pass

Planning to recover files for 295 folders...


Recovering files: 100%|██████████| 295/295 [2:17:07<00:00, 27.89s/it]  
